In [1]:
import torch
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch import loggers as pl_loggers
import numpy as np

from models.vqvae import VQVAE
from models.resnet import ResNet

In [2]:
class LatentDataset(torch.utils.data.Dataset):
    def __init__(self, path: str):
        super(LatentDataset, self).__init__()

        self.path = path

        self.data = np.load(self.path)

    def __len__(self):
        return self.data.shape[0] - 4

    def __getitem__(self, idx):
        
        item_1 = self.data[idx]
        item_2 = self.data[idx + 1]
        item_3 = self.data[idx + 2]
        item_4 = self.data[idx + 3]
        item_5 = self.data[idx + 4]

        return item_1, item_2, item_3, item_4, item_5

In [3]:
class LatentNetwork(pl.LightningModule):
    def __init__(self, lr: float = 4e-3, vqvae: VQVAE = None):
        super(LatentNetwork, self).__init__()

        self.lr = lr
        self.example_input_array = torch.rand(1, 8, 16, 8)

        self.vqvae = vqvae
        self.quantize_loss_weight = 0.1

        self.network = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=8, kernel_size=3, padding=1),
        )

        # self.network = ResNet(in_channels=8)

    def forward(self, x):

        out = self.network(x)

        return out
    
    def training_step(self, batch, batch_idx):
        x_1, x_2, x_3, x_4, x_5 = batch

        pred_1 = self(x_1)
        pred_1_quantized, quantize_loss_1, _ = self.vqvae.quantize(pred_1)

        pred_2 = self(pred_1_quantized)
        pred_2_quantized, quantize_loss_2, _ = self.vqvae.quantize(pred_2)

        pred_3 = self(pred_2_quantized)
        pred_3_quantized, quantize_loss_3, _ = self.vqvae.quantize(pred_3)

        pred_4 = self(pred_3_quantized)
        pred_4_quantized, quantize_loss_4, _ = self.vqvae.quantize(pred_4)

        prediction_loss = torch.nn.functional.mse_loss(pred_1_quantized, x_2) + torch.nn.functional.mse_loss(pred_2_quantized, x_3) + torch.nn.functional.mse_loss(pred_3_quantized, x_4) + torch.nn.functional.mse_loss(pred_4_quantized, x_5)
        commitment_loss = quantize_loss_1["commitment_loss"] + quantize_loss_2["commitment_loss"] + quantize_loss_3["commitment_loss"] + quantize_loss_4["commitment_loss"]

        loss = prediction_loss + self.quantize_loss_weight * commitment_loss

        self.log('train_loss', loss, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x_1, x_2, x_3, x_4, x_5 = batch

        pred_1 = self(x_1)
        pred_1_quantized, quantize_loss_1, _ = self.vqvae.quantize(pred_1)

        pred_2 = self(pred_1_quantized)
        pred_2_quantized, quantize_loss_2, _ = self.vqvae.quantize(pred_2)

        pred_3 = self(pred_2_quantized)
        pred_3_quantized, quantize_loss_3, _ = self.vqvae.quantize(pred_3)

        pred_4 = self(pred_3_quantized)
        pred_4_quantized, quantize_loss_4, _ = self.vqvae.quantize(pred_4)

        prediction_loss = torch.nn.functional.mse_loss(pred_1_quantized, x_2) + torch.nn.functional.mse_loss(pred_2_quantized, x_3) + torch.nn.functional.mse_loss(pred_3_quantized, x_4) + torch.nn.functional.mse_loss(pred_4_quantized, x_5)
        commitment_loss = quantize_loss_1["commitment_loss"] + quantize_loss_2["commitment_loss"] + quantize_loss_3["commitment_loss"] + quantize_loss_4["commitment_loss"]

        loss = prediction_loss + self.quantize_loss_weight * commitment_loss

        self.log('val_loss', loss, on_epoch=True)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer


In [4]:
# Hyperparameters
batch_size = 128
lr = 1e-3

# Dataset
train_dataset = LatentDataset("latent_test2.npy")
val_dataset = LatentDataset("latent_test2.npy")

# DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Model
checkpoint_vqvae = 'logs/vqvae_5channel_full_ds_32_64_128_128/c_1024/checkpoints/epoch=13-step=31727.ckpt'
vqvae = VQVAE.load_from_checkpoint(checkpoint_vqvae, in_channels=5)

model = LatentNetwork(lr=lr, vqvae=vqvae)
# model = LatentNetwork.load_from_checkpoint("logs/forecastnet/version_12/checkpoints/epoch=39-step=2880.ckpt", lr=1e-5, vqvae=vqvae)

In [5]:
def train():

    tb_logger = pl_loggers.TensorBoardLogger(
        save_dir="logs/", name="forecastnet")

    trainer = pl.Trainer(
        logger=tb_logger,
        max_epochs=40,
    )

    trainer.fit(
        model=model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )

train()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode  | In sizes      | Out sizes    
-------------------------------------------------------------------------------
0 | vqvae   | VQVAE      | 5.9 M  | train | ?             | ?            
1 | network | Sequential | 120 K  | train | [1, 8, 16, 8] | [1, 8, 16, 8]
-------------------------------------------------------------------------------
6.0 M     Trainable params
0         Non-trainable params
6.0 M     Total params
24.178    Tota

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\hendr\miniconda3\envs\pytorch_light\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


c:\Users\hendr\miniconda3\envs\pytorch_light\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 39: 100%|██████████| 72/72 [00:02<00:00, 28.15it/s, v_num=15]

`Trainer.fit` stopped: `max_epochs=40` reached.


Epoch 39: 100%|██████████| 72/72 [00:02<00:00, 26.62it/s, v_num=15]


In [6]:
model = LatentNetwork.load_from_checkpoint("logs/forecastnet/version_8/checkpoints/epoch=39-step=2880.ckpt", lr=1e-5, vqvae=vqvae).cpu()
model.eval()

sample_x, sample_y = train_dataset[0]

pred = model(torch.tensor(sample_x).unsqueeze(0).float())
pred, _, _ = model.vqvae.quantize(pred)

pred = pred.detach().numpy().squeeze()

from matplotlib import pyplot as plt

plt.imshow(sample_x[0].T, origin='lower')
plt.show()

plt.imshow(sample_y[0].T, origin='lower')
plt.show()

plt.imshow(pred[0].T, origin='lower')
plt.show()

mse = np.mean((sample_y - pred) ** 2)
print(mse)

ValueError: too many values to unpack (expected 2)